In [ ]:
!pip install -q ultralytics

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from PIL import Image

In [ ]:
df = pd.read_csv("/kaggle/input/weed-obj-detection/train.csv")

bboxes = df["bbox"].str.strip("[]").str.split(",", expand=True).astype(float)
bboxes.columns = ["x", "y", "w", "h"]
df = pd.concat([df, bboxes], axis=1)

df.head()

In [ ]:
image_ids = df.image_id.unique()
np.random.seed(42)
np.random.shuffle(image_ids)

split_idx = int(0.9 * len(image_ids))
train_ids = set(image_ids[:split_idx])
valid_ids = set(image_ids[split_idx:])

df["split"] = df.image_id.apply(
    lambda x: "train" if x in train_ids else "valid"
)

df.split.value_counts()

In [ ]:
BASE_DIR = "/kaggle/working/convertor/fold0"

for s in ["train", "valid"]:
    os.makedirs(f"{BASE_DIR}/images/{s}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/labels/{s}", exist_ok=True)

In [ ]:
IMG_DIR = "/kaggle/input/weed-obj-detection/train"

def to_yolo(row, img_w, img_h):
    xc = (row.x + row.w / 2) / img_w
    yc = (row.y + row.h / 2) / img_h
    return xc, yc, row.w / img_w, row.h / img_h

for img_id, g in df.groupby("image_id"):
    split = g.iloc[0]["split"]
    img_path = f"{IMG_DIR}/{img_id}.jpg"

    img = Image.open(img_path)
    w, h = img.size

    # copy image
    shutil.copy(img_path, f"{BASE_DIR}/images/{split}/{img_id}.jpg")

    # write label
    with open(f"{BASE_DIR}/labels/{split}/{img_id}.txt", "w") as f:
        for _, r in g.iterrows():
            xc, yc, bw, bh = to_yolo(r, w, h)
            f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

In [ ]:
import os
import shutil
from glob import glob
from tqdm.auto import tqdm

SRC_TEST = "/kaggle/input/weed-obj-detection/test"
DST_TEST = "/kaggle/working/convertor/fold0/images/test"

os.makedirs(DST_TEST, exist_ok=True)

test_imgs = glob(f"{SRC_TEST}/*")

for img_path in tqdm(test_imgs, desc="Copying test images"):
    fname = os.path.basename(img_path)
    dst_path = os.path.join(DST_TEST, fname)

    # avoid re-copying if cell is re-run
    if not os.path.exists(dst_path):
        shutil.copy(img_path, dst_path)

print(f"✅ Total test images in fold0: {len(os.listdir(DST_TEST))}")


In [ ]:
yaml_text = """train: /kaggle/working/convertor/fold0/images/train/
val: /kaggle/working/convertor/fold0/images/valid/
nc: 1
names: ['wheat']"""

with open("wheat.yaml", 'w') as f:
    f.write(yaml_text)

%cat wheat.yaml

In [ ]:
# Train
!yolo detect train \
    model=yolov10x.pt \
    data=wheat.yaml \
    imgsz=1024 \
    epochs=30 \
    batch=4 \
    optimizer=AdamW \
    mosaic=1.0 \
    mixup=0.3 \
    device=0 \
    project=/kaggle/working/wheat_baseline \
    name=yolov10x_baseline

# Keep one canonical checkpoint path for later cells
BEST_MODEL_PATH = "/kaggle/working/wheat_baseline/yolov10x_baseline/weights/best.pt"
print("BEST_MODEL_PATH:", BEST_MODEL_PATH)



In [ ]:
import os

assert os.path.exists(BEST_MODEL_PATH), f"❌ Missing checkpoint: {BEST_MODEL_PATH}"
print("✅ Found checkpoint:", BEST_MODEL_PATH)

!yolo detect val \
    model={BEST_MODEL_PATH} \
    data=wheat.yaml \
    imgsz=1024



In [ ]:
import os

# auto-detect test directory
CANDIDATES = [
    "/kaggle/input/global-wheat-detection/test_images",
    "/kaggle/input/weed-obj-detection/test",
    "/kaggle/working/convertor/fold0/images/test",
]

TEST_DIR = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert TEST_DIR is not None, "❌ Test directory not found!"
print("✅ Using test directory:", TEST_DIR)

!yolo detect predict \
  model={BEST_MODEL_PATH} \
  source={TEST_DIR} \
  imgsz=1024 \
  conf=0.25 \
  iou=0.6 \
  save_txt=True \
  save_conf=True \
  project=/kaggle/working/runs/detect \
  name=predict



In [ ]:
!ls runs/detect
!ls runs/detect/predict/labels | head

In [ ]:
import glob
import os

# predictions are written to <project>/<name>/labels when save_txt=True
pred_label_candidates = sorted(glob.glob('/kaggle/working/runs/detect/predict*/labels'))
assert pred_label_candidates, "❌ No prediction label folder found under /kaggle/working/runs/detect"

PRED_DIR = pred_label_candidates[-1]  # newest predict run
print("✅ Using prediction label directory:", PRED_DIR)



In [ ]:
import os
import pandas as pd
from PIL import Image

records = []

for file in os.listdir(PRED_DIR):
    if not file.endswith('.txt'):
        continue

    image_id = os.path.splitext(file)[0]

    # support common image extensions
    image_path = None
    for ext in ('.jpg', '.jpeg', '.png'):
        candidate = os.path.join(TEST_DIR, image_id + ext)
        if os.path.exists(candidate):
            image_path = candidate
            break
    if image_path is None:
        continue

    W, H = Image.open(image_path).size

    with open(os.path.join(PRED_DIR, file)) as f:
        for line in f:
            cls, xc, yc, w, h, conf = map(float, line.split())
            bw = w * W
            bh = h * H
            x = xc * W - bw / 2
            y = yc * H - bh / 2
            records.append({
                'image_id': image_id,
                'PredictionString': f"{conf:.4f} {x:.1f} {y:.1f} {bw:.1f} {bh:.1f}",
            })

df = pd.DataFrame(records)
print('Pred rows:', len(df))
print(df.head())



In [ ]:
print("Total test images:", len(os.listdir(TEST_IMG_DIR)))
print("Submission rows:", len(submission))